In [1]:
import pandas as pd

df = pd.read_csv("../../../Data/general_knowledge_qa.csv")
df.drop(columns=['question_type','image'],inplace=True)
df

,question,answer
0,How many days do we have in a week?,Seven
1,How many days are there in a normal year?,365 (not a leap year)
2,How many colors are there in a rainbow?,7
3,Which animal is known as the ‘Ship of the Dese...,Camel
4,How many letters are there in the English alph...,26
...,...,...
925,Where are the HeadQuarters of UNESCO?,"Paris, France"
926,Which are the colors of the rings of the Olymp...,"Black, Red, Blue, Green, and Yellow"
927,Who invented ‘Lift’?,Elisha Graves Otis
928,Which river crosses the equator twice?,Congo River


# 1. Tokenization

In [2]:
def tokenize(text):
    return (text.lower()
            .replace("?","")
            .replace("!","")
            .replace("'","")
            .replace(",","")
            .replace(".","")
            .replace(":","")
            .replace("(","")
            .replace(")","")
            .split())

In [3]:
tokenize("How many days do we have in a week?")

['how', 'many', 'days', 'do', 'we', 'have', 'in', 'a', 'week']

# 2. Build vocab

In [4]:
vocab = {
    "UNK":0
}

In [5]:
def build_vocab(row):
    question_tokens = tokenize(row['question'])
    answer_tokens = tokenize(row['answer'])
    
    merge_tokens = question_tokens + answer_tokens
    
    for token in merge_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

In [6]:
build_vocab(df.iloc[0,:])
vocab

{'UNK': 0,
 'how': 1,
 'many': 2,
 'days': 3,
 'do': 4,
 'we': 5,
 'have': 6,
 'in': 7,
 'a': 8,
 'week': 9,
 'seven': 10}

In [7]:
df.apply(build_vocab, axis=1)  # every row at a time
print(len(vocab))
vocab

2154


{'UNK': 0,
 'how': 1,
 'many': 2,
 'days': 3,
 'do': 4,
 'we': 5,
 'have': 6,
 'in': 7,
 'a': 8,
 'week': 9,
 'seven': 10,
 'are': 11,
 'there': 12,
 'normal': 13,
 'year': 14,
 '365': 15,
 'not': 16,
 'leap': 17,
 'colors': 18,
 'rainbow': 19,
 '7': 20,
 'which': 21,
 'animal': 22,
 'is': 23,
 'known': 24,
 'as': 25,
 'the': 26,
 '‘ship': 27,
 'of': 28,
 'desert’': 29,
 'camel': 30,
 'letters': 31,
 'english': 32,
 'alphabet': 33,
 '26': 34,
 'consonants': 35,
 '21': 36,
 'sides': 37,
 'triangle': 38,
 'three': 39,
 'month': 40,
 'has': 41,
 'least': 42,
 'number': 43,
 'february': 44,
 'vowels': 45,
 'series': 46,
 'e': 47,
 'i': 48,
 'o': 49,
 'u': 50,
 'called': 51,
 'king': 52,
 'jungle': 53,
 'lion': 54,
 'primary': 55,
 'red': 56,
 'yellow': 57,
 'blue': 58,
 '29': 59,
 'what': 60,
 'you': 61,
 'call': 62,
 'house': 63,
 'made': 64,
 'ice': 65,
 'igloo': 66,
 'largest': 67,
 'world': 68,
 'whale': 69,
 'tallest': 70,
 'on': 71,
 'earth': 72,
 'giraffe': 73,
 'festival': 74,
 'ho

# 3. Text to numeric representation

In [8]:
def text_to_numeric_rep(text, vocab):
    numerical_present = []
    for word in tokenize(text):
        if word in vocab:
            numerical_present.append(vocab[word])
        else:
            numerical_present.append(0)
    return numerical_present

In [10]:
text_to_numeric_rep("Which animal is known as the ‘Ship of the Desert?’",vocab)

[21, 22, 23, 24, 25, 26, 27, 28, 26, 29]

# 4. Dataloader

In [12]:
from torch.utils.data import Dataset, DataLoader
import torch

class CustomDataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        row = self.df.iloc[index]

        numerical_question = text_to_numeric_rep(row['question'], self.vocab)
        numerical_answer = text_to_numeric_rep(row['answer'], self.vocab)
        
        return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [13]:
dataset = CustomDataset(df,vocab)

dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [14]:
for ques, ans in dataloader:
    print(ques)
    print(ans)
    break

tensor([[496,  23, 121, 497,  81, 421, 266, 422]])
tensor([[422, 496,  23, 121, 498,  81]])


# 5. Model Build

In [15]:
import torch.nn as nn
class QuesAnsRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50,64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)
        
    def forward(self, question):
        embedding_ques = self.embedding(question)
        hidden_output, final_output = self.rnn(embedding_ques)
        return self.fc(final_output.squeeze(0))

# 6. Train Model

In [16]:
learning_rate = 0.001
epochs = 50

In [17]:
model = QuesAnsRNN(len(vocab))

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [18]:
for epoch in range(epochs):
    epochs_loss = 0
    for question, answer in dataloader:
        optimizer.zero_grad()
        
        y_pred = model(question)
        
        loss = criterion(y_pred, answer[:,0])
        
        loss.backward()
        
        optimizer.step()
        
        epochs_loss += loss.item()
        
    avg_epoch_loss = epochs_loss / len(dataloader)
    print(f"Epoch {epoch+1}: Loss {avg_epoch_loss}")

Epoch 1: Loss 7.329386776172987
Epoch 2: Loss 5.480903202231213
Epoch 3: Loss 4.342822493949244
Epoch 4: Loss 3.3745112895324665
Epoch 5: Loss 2.5285463999355993
Epoch 6: Loss 1.8191203835510439
Epoch 7: Loss 1.2674941828133919
Epoch 8: Loss 0.8771150904998023
Epoch 9: Loss 0.6166899949081883
Epoch 10: Loss 0.44575684500097107
Epoch 11: Loss 0.3275779057148924
Epoch 12: Loss 0.25037632546918365
Epoch 13: Loss 0.18397448181840642
Epoch 14: Loss 0.1329133028620153
Epoch 15: Loss 0.09781307408658747
Epoch 16: Loss 0.0710122559342273
Epoch 17: Loss 0.04994654023108543
Epoch 18: Loss 0.04083431987210818
Epoch 19: Loss 0.10746499835588638
Epoch 20: Loss 0.1033110438247702
Epoch 21: Loss 0.036111443005602366
Epoch 22: Loss 0.023334143961991325
Epoch 23: Loss 0.02321074938985847
Epoch 24: Loss 0.018133247635854277
Epoch 25: Loss 0.016820350746314043
Epoch 26: Loss 0.01597066065277754
Epoch 27: Loss 0.015113115249856023
Epoch 28: Loss 0.015447562201484044
Epoch 29: Loss 0.020019990508886643
Epo

# 7. Inference

In [33]:
def predict(model, question, threshold=0.5, words=1):
    # convert text to number
    numerical_question = text_to_numeric_rep(question, vocab)
    
    # make it tensor
    tensor_question = torch.tensor(numerical_question).unsqueeze(0)
    
    # probabilities
    output_probability = model(tensor_question)
    
    # use softmax
    probs = torch.nn.functional.softmax(output_probability, dim=1)
    
    # get max probs
    max_prob, max_index = torch.max(probs, dim=1)
    
    inverse_vocab = {value:key for key, value in vocab.items()}
    
    if max_prob >= threshold:
        print(list(vocab.keys())[max_index])
        print(f"Probability: {max_prob}")
    else:
        print("I dont know")
        print(f"Probability: {max_prob}")
    

In [34]:
predict(model,"How many days are there in a normal year?",words=1, threshold=0.5)

365
Probability: tensor([0.9985], grad_fn=<MaxBackward0>)


In [36]:
predict(model,"Which animal is known as the ‘Ship of the Desert?’",words=1, threshold=0.8)

camel
Probability: tensor([0.9999], grad_fn=<MaxBackward0>)


In [42]:
predict(model,"How many sides are there in a triangle?’",words=1, threshold=0.7)

I dont know
Probability: tensor([0.5207], grad_fn=<MaxBackward0>)


In [20]:
df

,question,answer
0,How many days do we have in a week?,Seven
1,How many days are there in a normal year?,365 (not a leap year)
2,How many colors are there in a rainbow?,7
3,Which animal is known as the ‘Ship of the Dese...,Camel
4,How many letters are there in the English alph...,26
...,...,...
925,Where are the HeadQuarters of UNESCO?,"Paris, France"
926,Which are the colors of the rings of the Olymp...,"Black, Red, Blue, Green, and Yellow"
927,Who invented ‘Lift’?,Elisha Graves Otis
928,Which river crosses the equator twice?,Congo River
